In [21]:
import warnings
warnings.filterwarnings('ignore')

# Lab 4 - Data (ETL)

In [22]:
%matplotlib inline

## General Instructions

In this course, Labs are the chance to applying concepts and methods discussed in the module.
They are a low stakes (pass/fail) opportunity for you to try your hand at *doing*.
Please make sure you follow the general Lab instructions, described in the Syllabus.
The summary is:

* Discussions should start as students work through the material, first Wednesday at the start of the new Module week. 
* Labs are due by Sunday. 
* Lab solutions are released Monday.  
* Post Self Evaluation and Lab to Lab Group on Blackboard and Lab to Module on Blackboard on Monday.

The last part is important because the Problem Sets will require you to perform the same or similar tasks without guidance.
Problem Sets are your opportunity to demonstrate that you understand how to apply the concepts and methods discussed in the relevant Modules and Labs.

## Specific Instructions

1.  For Blackboard submissions, if there are no accompanying files, you should submit *only* your notebook and it should be named using *only* your JHED id: fsmith79.ipynb for example if your JHED id were "fsmith79". If the assignment requires additional files, you should name the *folder/directory* your JHED id and put all items in that folder/directory, ZIP it up (only ZIP...no other compression), and submit it to Blackboard.

    * do **not** use absolute paths in your notebooks. All resources should located in the same directory as the rest of your assignments.
    * the directory **must** be named your JHED id and **only** your JHED id.
    * do **not** return files provided by us (data files, .py files)

2. Data Science is as much about what you write (communicating) as the code you execute (researching). In many places, you will be required to execute code and discuss both the purpose and the result. Additionally, Data Science is about reproducibility and transparency. This includes good communication with your team and possibly with yourself. Therefore, you must show **all** work.

3. Avail yourself of the Markdown/Codecell nature of the notebook. If you don't know about Markdown, look it up. Your notebooks should not look like ransom notes. Don't make everything bold. Clearly indicate what question you are answering.

4. Submit a cleanly executed notebook. The first code cell should say `In [1]` and each successive code cell should increase by 1 throughout the notebook.

**Note** This assignment will have multiple files. Follow those instructions.

## Lab

**Reid's** is a small breakfast stand that sells drinks (coffee, tea, sodas) and food (egg & sausage, oatmeal) in a commercial downtown area, Monday through Friday, from 8a until 11am.
Although their menu is small, they do try to cater to a wide variety of diets and thus provide both vegan and keto options for most of their meals.
They started using Ordr as their Point of Sale (POS) system about two months ago and are on the Basic Plan.

Under the Basic Plan, they are able to use the Ordr API to access orders.
This order information comes in the form of a denormalized JSON document.
In order to make any sense of things, you need to normalize it in the Datawarehouse.

1. You are not actually going to access an external API. Use the provided JSON file as the data that the API would return.
2. **You are doing "ETL in the Large" in this assignment.** You are going to build a datawarehouse in SQLite, *not* an application database. This difference is substantial. Refer to the draft chapter of Fundamentals for some of the differences.

**Note** We sometimes get strange questions about the use of SQLite like, "do you really use SQLite in production?". We use SQLite jor this Lab for the following reasons:

1. SQLite is a real RDBMS.
2. SQLite uses real SQL. SQL may be the most important skill you can have as a Data Scientist doing Data Science.
3. Most importantly, the database itself is a standalone file that you can submit to us.

That being said, under some and somewhat weird circumstances, I have used SQLite on real projects before. However, the learning objective is not SQLite, SQLite is  tool.

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Note</strong>
    <p>You may need to install <tt>sqlite</tt>. It is normally on MacOs and may be on Linux already.</p>
</div>

**Note**
We assume you know the basics of RDMBS and SQL DDL in this course (It is in the course prerequisites!).
That you understand what "normalized" and "denormalized" data means and that you know about primary and foreign keys.
This [article](https://www3.ntu.edu.sg/home/ehchua/programming/sql/Relational_Database_Design.html) does talk about the major points.
Additionally, we assume you know SQL and DDL.
If you do not, this Lab will be more challenging than usual and you should start early.

**Important - You must not use Pandas for any part of this assignment.**
Why not?
Because you should know how to do these things without relying on Pandas.

## Part 1

### Learning Objectives

* investigate the structure of data acquired from a 3rd party.
* convert denormalized data into normalized data, according to common data warehouse practices.
* design a data warehouse to store production data acquired from a 3rd party.
* write data to a data warehouse.

This assignment is not about tools *per se* but about broader skills and concepts.

You will be creating the following files:

1. **reids.sql** - this file will create the database structure using DDL. Make sure you review the data and sketch out your design.
2. **reids.db** - this is the actual database, our data warehouse.

```
> sqlite3 reids.db < reids.sql
```

will create the database and all the tables.
The database will be empty at this point.

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Note</strong>
    <p>"<tt>></tt>" represents the command line. Your sqlite executable may have a different name.</p>
</div>

3. **reids.py** - this program will parse the JSON file and fill the database.

```
> python reids.py
```

Unfortunately, the documentations for the Ordr API is sparse, here is an example of one order:

```
{'items': [
    {'name': 'coffee', 'price': 2.75},
    {'name': 'flavor shot', 'price': 1.0}
    ],
'charges': {
    'date': '01/04/21 10:22',
    'subtotal': 3.75,
    'taxes': 0.26,
    'total': 4.01},
'payment': {
    'card_type': 'visa',
    'last_4_card_number': '0465',
    'zip': '21217',
    'cardholder': 'Christina Sampson',
    'method': 'credit_card'}}
```

**The date format is Month/Day/Year**

Make sure you look through the data to see what values are possible for each of the fields.
The standards of normalization/denormalization for datawarehouses are slightly different or can be different than regular production RDMBS systems.
For example, we might be tempted to create a `menu_items` table:

```
id    name                 price
1     coffee               2.75
2     flavor shot          1.00
3     egg salad sandwich   4.50
```

An issue arises if we change the name to "Kona Coffee" because it will change it *for past purchases*.
That is, customers in the past bought based on the name "Coffee" and not "Kona Coffee".
This might be important.
Even worse, if we change the price to \\$3.00, it changes it for all past purchases and that is clearly wrong.

In a production RDMBS we often want the data to change everywhere it is used.
If "Steve" changes his name to "Sam", we want that to be reflected in any query and report.
For datawarehousing, though, we want to preserve the historical fidelity of the data.
This means we have a tendency to normalize *less* than we would otherwise do.
It's worth noting that there is a trend to preserve the historical fidelity in production databases as well by things like soft deletes.

This means the main issue for the Ordr data is storing the three main entities and creating primary/secondary keys.
You will need to create these.

All of this "parsing and massaging" work will be done in the `reids.py` file.
It will contain the code to parse the JSON file and fill the database, performing whatever normalization and standardization is required as well as creating whatever primary and foreign key relationships seem reasonable.

You must create the following tables in the database:

1. `items`
2. `charges`
3. `payments`

but you can add additional tables as necessary (it is not uncommmon to include tables in datawarehouses that support analytics such as information about business dates).

**Note** Feel free to develop reids.py as a Notebook and then generate the .py from the .ipynb file...just make sure you only include the .py file and that it will run from the command line as specified above and you have commented out any debug/chatter.

**Important**
There are some "gotchas".
1. When inserting data into the database, don't forget to the commit.
2. If you must reconstruct your database, make sure you "free" all references to it. If you use a script to change it but it's open your notebook, the open version in the notebook won't necessarily see those changes.  You'll need to get a new connection.

When you are done with this part, you should be able to proceed to Part 2.

**Everything having to do with parsing the JSON file from Ordr and setting up the "data warehouse" should be done in the three files described above and not in this Notebook.**

## Part 2

### Learning Objectives

You almost never start out with a Notebook and start pulling data. 
The idea that you launch Jupyter Notebook and load a readily available CSV is an incredibly artificial artifact of school (if you had to pull data The Real Way(tm) for every assignment, we'd never get anything done).

Instead, you are more likely to start out with a database and you run queries directly against the database, finding out where and what everything is, answering some initial questions.

* Run queries against an RDBMS to answer basic business questions.

Some data science projects are literally just this: someone asks a question, you investigate the data, you run a query using something like [MySQL Workbench](https://www.mysql.com/products/workbench/), [Toad](https://www.toadworld.com/products/toad-for-sql-server) (Windows Only) or [Postico](https://eggerapps.at/postico/) (MacOS Only). There are also generic SQL clients. For example, [VSCode](https://code.visualstudio.com/) has SQL extensions.

You will mimic that experience here by using only the [sqlite3](https://docs.python.org/3/library/sqlite3.html) Python library (included in the base installiation, link is to documentation).
As with Part 1, you may *not* use Pandas for this part.
Additionally, you *must* not print out native Python data structures.
[Tabulate](https://pypi.org/project/tabulate/) has been provided in the environment.yml for your use.


For Part 2, everything should be done here, in this notebook.

**Note**
The general format is discuss/code/discuss.
For the questions below, you should be able to:

1. explain what the query does (discuss)
2. execute and display the query result (code)
3. interpret the result (discuss)

All three are required for full credit on something like Problem Set so you should practice the triad here. It is permissible to use a query to get raw data (and show it in a table) and then perform a calculation with that raw data (just add a code cell). However, you should do as much as possible in SQL.

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Note</strong>
    <p>
There is a significant work/payoff imbalance here and this reflects real life.
Setting up the data warehouse is 80% of the effort but only 20% of the credit.
Your boss just doesn't care about your struggles with the data, they only care about answering the queries.
As a result, the queries (Part 2) may be 20% of the effort but they're 80% of the grade.
They're proof that you did Part 1 correctly.
If you don't get to the queries, if you don't do them right, there's no proof.
    </p>
</div>

Using the database `reids.db` and SQL please answer the following questions:

In [23]:
from tabulate import tabulate
import sqlite3

In [24]:
from pathlib import Path
cwd = Path.cwd()
con = sqlite3.connect('redis.db')

### Question 1.

What were Reid's order count and gross revenue by day for the two month period?

SQL Explanation
- Q1: Order count and gross revenue by day




In [25]:
query = '''
SELECT
    DATE(o.order_date)          AS day,
    COUNT(o.order_id)           AS order_count,
    ROUND(SUM(oc.total), 2)     AS gross_revenue
FROM orders o
JOIN order_charges oc ON o.order_id = oc.order_id
GROUP BY day
ORDER BY day;
'''

cursor = con.cursor()
cursor.execute(query)
rows = cursor.fetchall()

print(tabulate(rows, headers=['Day', 'Order Count', 'Gross Revenue'], tablefmt='psql'))

+------------+---------------+-----------------+
| Day        |   Order Count |   Gross Revenue |
|------------+---------------+-----------------|
| 2021-04-01 |            34 |          188.87 |
| 2021-04-02 |            51 |          265.11 |
| 2021-04-05 |            57 |          339.77 |
| 2021-04-06 |            48 |          276.09 |
| 2021-04-07 |            32 |          188.63 |
| 2021-04-08 |            58 |          345.92 |
| 2021-04-09 |            62 |          341.58 |
| 2021-04-12 |            55 |          294.54 |
| 2021-04-13 |            32 |          167.5  |
| 2021-04-14 |            54 |          288.41 |
| 2021-04-15 |            45 |          262.69 |
| 2021-04-16 |            75 |          402.63 |
| 2021-04-19 |            49 |          276.1  |
| 2021-04-20 |            43 |          265.65 |
| 2021-04-21 |            39 |          207.62 |
| 2021-04-22 |            55 |          316.76 |
| 2021-04-23 |            75 |          408.49 |
| 2021-04-26 |      

__Q1 Discussion__

Daily orders ranged from 30 to 75, and revenue from $167 to $408. There's definitely day-to-day variance even within the same week suggesting demand isn't just routine, there may be weather, local events, or promotions factoring into the fluctuations. 



### Question 2.

What is Reid's average order count and gross revenue by day of the week?


SQL Explanation
- Q2: Average order count and gross revenue by day of week
    - Uses a CTE to first aggregate to one row per calendar day, then averages across all occurrences of each weekday.
    - Note: the shop seems to be closed on weekends (no data for Sat/Sun), so only Mon-Fri appear in results.




In [26]:
query = '''
WITH daily AS (
    SELECT
        DATE(o.order_date)                              AS day,
        CAST(strftime('%w', o.order_date) AS INTEGER)   AS dow_num,
        CASE strftime('%w', o.order_date)
            WHEN '0' THEN 'Sunday'
            WHEN '1' THEN 'Monday'
            WHEN '2' THEN 'Tuesday'
            WHEN '3' THEN 'Wednesday'
            WHEN '4' THEN 'Thursday'
            WHEN '5' THEN 'Friday'
            WHEN '6' THEN 'Saturday'
        END                                             AS day_of_week,
        COUNT(o.order_id)                               AS order_count,
        SUM(oc.total)                                   AS revenue
    FROM orders o
    JOIN order_charges oc ON o.order_id = oc.order_id
    GROUP BY day
)
SELECT
    day_of_week,
    ROUND(AVG(order_count), 1)  AS avg_orders,
    ROUND(AVG(revenue), 2)      AS avg_revenue
FROM daily
GROUP BY dow_num, day_of_week
ORDER BY dow_num;
'''

cursor = con.cursor()
cursor.execute(query)
rows = cursor.fetchall()

print(tabulate(rows, headers=['Day of Week', 'Avg Orders', 'Avg Revenue'], tablefmt='psql'))

+---------------+--------------+---------------+
| Day of Week   |   Avg Orders |   Avg Revenue |
|---------------+--------------+---------------|
| Monday        |         48.1 |        276.63 |
| Tuesday       |         43.6 |        247.23 |
| Wednesday     |         41.4 |        232.76 |
| Thursday      |         51.4 |        292.97 |
| Friday        |         60.3 |        325.24 |
+---------------+--------------+---------------+


__Q2 Discussion__

Friday is clearly the strongest day with about 45% more orders than Wednesday. The mid-week decline (Tue–Wed) seems common for the food industry as a whole. It may be beneficial for Reid's to consider promotions on slow days or ensure they're fully staffed on Fridays. The shop appears to be closed on weekends, as there is no Saturday or Sunday data.





### Question 3.

How many cups of coffee does Reid's sell per day, on average?


SQL Explanation
- Q3: Average cups of coffee sold per day
    - Counts rows in order_items where name = 'coffee', grouped by day, then averages across all days.




In [27]:
query = '''


WITH daily_coffee AS (
    SELECT
        DATE(o.order_date)  AS day,
        COUNT(i.item_id)    AS cups
    FROM orders o
    JOIN order_items i ON o.order_id = i.order_id
    WHERE i.name = 'coffee'
    GROUP BY day
)
SELECT ROUND(AVG(cups), 1) AS avg_cups_per_day
FROM daily_coffee;
'''

cursor = con.cursor()
cursor.execute(query)
rows = cursor.fetchall()

print(tabulate(rows, headers=['Avg Cups of Coffee per Day'], tablefmt='psql'))

+------------------------------+
|   Avg Cups of Coffee per Day |
|------------------------------|
|                         36.1 |
+------------------------------+


__Q3 Discussion__

Reid's sells an average of 36.1 cups of coffee per day. Given the average order count of about 49 orders/day, this means roughly 75% of orders include a coffee which is consistent with the shop being primarily focused on coffee.

### Question 4.

What proportion of orders contain "up charges" like flavor shots, vegan or keto substitutions?


SQL Explanation
- Q4: Proportion of orders containing an up-charge
    - Up-charges identified from the item catalog: 'flavor shot', 'vegan', 'keto'




In [28]:
query = '''
WITH upcharge_orders AS (
    SELECT DISTINCT order_id
    FROM order_items
    WHERE name IN ('flavor shot', 'vegan', 'keto')
)
SELECT
    COUNT(DISTINCT uo.order_id)                         AS upcharge_orders,
    (SELECT COUNT(*) FROM orders)                       AS total_orders,
    ROUND(
        COUNT(DISTINCT uo.order_id) * 100.0 /
        (SELECT COUNT(*) FROM orders), 1
    )                                                   AS pct_with_upcharge
FROM upcharge_orders uo;
'''

cursor = con.cursor()
cursor.execute(query)
rows = cursor.fetchall()

print(tabulate(rows, headers=['Upcharge Orders', 'Total Orders', '% with Upcharge'], tablefmt='psql'))

+-------------------+----------------+-------------------+
|   Upcharge Orders |   Total Orders |   % with Upcharge |
|-------------------+----------------+-------------------|
|               430 |           2071 |              20.8 |
+-------------------+----------------+-------------------+


__Q4 Discussion__

20.8% of orders (430 out of 2,071) contain at least one up-charge (flavor shot, vegan, or keto substitution). This is a large share of the total orders where roughly 1 in 5 customers is customizing their order and paying extra for it. It is likely that these add-ons have high margins, so it would be beneficial for the shop to promote them more heavily.

### Question 5.

Reid's considers someone to be a "regular" if they come at least 3 out of 5 days per week. How many regulars do you estimate there are and what are their names? How many days per week do they each come on average? What are the limits of this calculation based on the available data?


SQL Explanation:
- Q5: Estimated regulars (>=3 visits in a 5 day work week)
    - Method:
    1. Identify unique person and day visits using cardholder name
    2. Group into calendar weeks (Mon-Fri) and count distinct
       visit days per customer per week.
    3. A customer is a "regular week" if they visited >=3 days.
    4. Flag as a regular if >=50% of their weeks are regular
       weeks, and they appear in at least 3 weeks total
       (filters out one off high visit customers).




In [29]:
query = '''


WITH card_visits AS (
    SELECT
        p.cardholder,
        DATE(o.order_date)              AS visit_day,
        strftime('%Y-%W', o.order_date) AS iso_week
    FROM orders o
    JOIN order_payments p ON o.order_id = p.order_id
    WHERE p.cardholder IS NOT NULL
    GROUP BY p.cardholder, visit_day        -- one visit per person per day
),
weekly AS (
    SELECT
        cardholder,
        iso_week,
        COUNT(DISTINCT visit_day)       AS days_that_week
    FROM card_visits
    GROUP BY cardholder, iso_week
),
customer_stats AS (
    SELECT
        cardholder,
        COUNT(DISTINCT iso_week)                                AS weeks_seen,
        SUM(days_that_week)                                     AS total_visit_days,
        SUM(CASE WHEN days_that_week >= 3 THEN 1 ELSE 0 END)   AS regular_weeks
    FROM weekly
    GROUP BY cardholder
)
SELECT
    cardholder,
    weeks_seen,
    total_visit_days,
    regular_weeks,
    ROUND(total_visit_days * 1.0 / weeks_seen, 2)   AS avg_days_per_week
FROM customer_stats
WHERE regular_weeks * 1.0 / weeks_seen >= 0.5
  AND weeks_seen >= 3
ORDER BY avg_days_per_week DESC;
'''

cursor = con.cursor()
cursor.execute(query)
rows = cursor.fetchall()

print(tabulate(rows, headers=['Cardholder', 'Weeks Seen', 'Total Visit Days', 'Regular Weeks', 'Avg Days/Week'], tablefmt='psql'))

+-------------------+--------------+--------------------+-----------------+-----------------+
| Cardholder        |   Weeks Seen |   Total Visit Days |   Regular Weeks |   Avg Days/Week |
|-------------------+--------------+--------------------+-----------------+-----------------|
| Anthony Martin    |            8 |                 34 |               8 |            4.25 |
| Gina Green        |            8 |                 34 |               8 |            4.25 |
| Rebecca Garza     |            9 |                 35 |               8 |            3.89 |
| Richard Young     |            8 |                 31 |               7 |            3.88 |
| Lisa Aguilar      |            9 |                 34 |               8 |            3.78 |
| Cassandra Francis |            9 |                 33 |               8 |            3.67 |
| Eric Bruce        |            9 |                 33 |               7 |            3.67 |
| Gregory Jones     |            9 |                 33 |   

__Q5 Discussion__

30 customers were identified as regulars, visiting an average of 3.0 to 4.25 days per week. The top regulars came in nearly every single workday.

The estimate does have limitations:
- Cash customers are not included. about 17% of orders have no cardholder name, so any regular who pays cash is missed meaning the true number of regulars could be higher.
- Name matching is not perfect. Two customers with the same name will get merged and a couple sharing one card will get counted as one person.
- Two months is a fairly short window. Some regulars may have been on vacation or visited inconsistently enough to fall below the threshold despite being frequent customers overall for the shop.



